# Phase 3: Data Cleaning & Preprocessing
In this notebook, we prepare the raw data for machine learning by handling categorical variables, scaling numerical features, and creating a clean dataset.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os

# Load the dataset
df = pd.read_csv('../data/raw/customer_details.csv')
print("Original shape:", df.shape)

In [ ]:
# 1. Handle Missing Values and Duplicates
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

# Drop duplicates if any
df = df.drop_duplicates()

In [ ]:
# 2. Drop Identifier columns (Not useful for ML)
if 'Customer ID' in df.columns:
    df = df.drop(['Customer ID'], axis=1)

In [ ]:
# 3. Binary Encoding for Yes/No columns
binary_cols = ['Subscription Status', 'Discount Applied', 'Promo Code Used']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

display(df[binary_cols].head())

In [ ]:
# 4. Ordinal Encoding for Size and Frequency
# Map sizes to numbers (S=1, M=2, L=3, XL=4)
size_mapping = {'S': 1, 'M': 2, 'L': 3, 'XL': 4}
df['Size'] = df['Size'].map(size_mapping)

# Map frequencies to numbers (Annually=1, Weekly=5)
freq_mapping = {
    'Annually': 1, 
    'Every 3 Months': 2, 'Quarterly': 2,
    'Monthly': 3,
    'Bi-Weekly': 4, 'Fortnightly': 4,
    'Weekly': 5
}
df['Frequency of Purchases'] = df['Frequency of Purchases'].map(freq_mapping)

In [ ]:
# 5. One-Hot Encoding for remaining categorical features
categorical_cols = ['Gender', 'Category', 'Season', 'Shipping Type', 'Payment Method']

# High-cardinality columns (like Location, Item Purchased, Color) are dropped 
# for now to avoid creating hundreds of columns in our baseline model.
df = df.drop(['Location', 'Item Purchased', 'Color'], axis=1)

# Get dummy variables (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", df_encoded.shape)

In [ ]:
# 6. Scale Numerical Features
# We use StandardScaler so all numerical inputs have a mean of 0 and variance of 1
scaler = StandardScaler()
num_cols = ['Age', 'Purchase Amount (USD)', 'Review Rating', 'Previous Purchases']
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

display(df_encoded.head())

In [ ]:
# 7. Save Processed Data
# Create directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)
processed_path = '../data/processed/customer_details_processed.csv'

df_encoded.to_csv(processed_path, index=False)
print(f"Cleaned and encoded data saved to {processed_path}")